## TIFF plot

In [ ]:
# Settings + Data fetching
system("conda install -y conda-forge::r-rcpp conda-forge::openssl conda-forge::r-sf conda-forge::r-terra conda-forge::r-ncdf4")
system("conda install -y conda-forge::r-r.utils conda-forge::r-tidyverse conda-forge::libgdal-hdf5 conda-forge::r-ggplot2")

# install.packages("ncdf4", "tidyverse", "terra", "dplyr", "sf", "jsonlite", "utils")
# install.packages("ncdf4")

# GET THE DEFAULT INITIAL PARAMETERS

## CONDA ENVIRONEMENT (OPTIONAL)
Sys.getenv("CONDA_DEFAULT_ENV")
## VERSION OF R IN USE
version

system("conda activate")

In [ ]:
library(ncdf4)
library(R.utils)
library(tidyverse) # because who can live without the tidyverse?
library(terra)     
library(dplyr)
library(sf)        
library(jsonlite) 
library(utils)
library(ggplot2)

In [ ]:
global_topo_tiff_gz <- "global_topo.tiff.gz"

# nchar(global_topo_tiff)
filename.length <- nchar(global_topo_tiff_gz)

# Get the extension ("." + 2 letters)
substr(global_topo_tiff_gz,filename.length-2, filename.length)

# Get the name without the extension ("." + 2 letters)
start <- 1
end <- filename.length-3
global_topo_tiff <- substr(global_topo_tiff_gz,start, end)

print(global_topo_tiff_gz)
print(global_topo_tiff)

# ==============================================================================

if(!file.exists(global_topo_tiff_gz)){
    download.file("https://topex.ucsd.edu/pub/global_topo_tiff/global.tiff.gz", global_topo_tiff_gz, mode = "wb")
}

# ==============================================================================
# UNZIP FILENAME
# ==============================================================================

if(!file.exists(global_topo_tiff)){
    R.utils::gunzip(global_topo_tiff_gz, overwrite=FALSE, remove=TRUE, BFR.SIZE=1e+07)
}

In [ ]:
# ==============================================================================
# FILE INFO
# ==============================================================================
print(file.info( substr(global_topo_tiff_gz,start, end) ))
print(file.info( substr(global_topo_tiff,start, end) ))

ls()

## NetCDF (Copernicus Data Files)

In [ ]:
system("conda install -y conda-forge::r-lubridate conda-forge::r-rcolorbrewer conda-forge::r-lattice conda-forge::r-png")

In [ ]:
library(ncdf4) #     ncdf4: open, write and create NetCDF files (also provides metadata information)
library(lubridate) # lubridate: operate on date and times data
library(RColorBrewer) # RColorBrewer: create colour palettes for thematic maps
library(lattice) # lattice : visualization system for typical graphics

In [ ]:
ls()

In [ ]:
# 2.2 DOWNLOAD AND LOAD BATHYMETRY ----
bathy_file <- "global_topo_1min_topo_19_1.nc"
if(!file.exists(bathy_file)){
    download.file("https://topex.ucsd.edu/pub/global_topo_1min/topo_19.1.nc", bathy_file, mode = "wb")
}

print('File Downloaded')

## Data Exploration

File exploration : ornldaac [github repository](https://github.com/ornldaac/netCDF_data_in_R/blob/master/netCDF_in_r_ornldaac_tutorial.md)

In [ ]:
## A.7 Draw Pairs Plot of Data Frame Columns
'install.packages("GGally")             # Install GGally package
library("GGally")                      # Load GGally package'

# ggpairs(Bathy)                        # Draw pairs plot

In [ ]:
## A.8 Boxplots of Multiple Columns 
# ggplot(as.data.frame(Bathy),                    # Draw boxplots
#        aes(x = value,
#            fill = name)) +
#   geom_boxplot()

In [ ]:
## A.9 Histograms of Multiple Columns 
# ggplot(Bathy,                    # Draw histograms
#        aes(x = value)) +
#   geom_histogram() + 
#   facet_wrap(name ~ ., scales = "free")

In [ ]:
# Open the NetCDF file

nc_file <- nc_open(bathy_file)
print(nc_file)

# Structure of the file

names(nc_file)
# Names of the variables
names(nc_file$var)
# Names of the dimensions
names(nc_file$dim)

In [ ]:
# List the attributes and sub-attributes

i <- 1
for(listVar in names(nc_file)){
    cat(i, listVar,"\n")
    for(listNames in names(nc_file[[listVar]])){
        cat("Attr:", listNames, ":", names(nc_file[[listVar]][[listNames]]),"\n")
    }
    i <- i + 1
}

In [ ]:
# Sub-Content for lat and lon
names(nc_file$dim$lon)
names(nc_file$dim$lat)

nc_file$dim$lon[1:5]
nc_file$dim$lat[1:5]

# 5 first Values
nc_file$dim$lon$vals[1:5] # print(nc_file$dim['lon'])
nc_file$dim$lat$vals[1:5]
nc_file$var$z$id[1:5]


In [ ]:
# Get coordinates variables
longitude <- ncvar_get(nc_file,"lon")
latitude <- ncvar_get(nc_file,"lat")
z <- ncvar_get(nc_file,"z")

In [ ]:
# Dimensions of latitude & longitude
print(c(length(longitude), length(latitude)))
print(dim(z))

# Check longtitude and latitude values
cat(" Head of longitude:",head(longitude),"\n")
cat(" Head of latitude:",head(latitude),"\n")

fillvalue <- ncatt_get(nc_file, "z", "_FillValue") # The fill value (aka, the no data value) is -9999.
print(fillvalue$value)

In [ ]:
nc_close(nc_file)

In [ ]:
z[z == fillvalue$value] <- NA
nc.slice.min80 <- z[,1]
dim(nc.slice.min80)

In [ ]:
r <- rast(nc.slice.min80,
  extent = ext(min(lon), max(lon), min(lat), max(lat)),
  crs = "+proj=longlat +ellps=WGS84 +datum=WGS84 +no_defs +towgs84=0,0,0"
)
rm(nc.slice.min80)

### Plots

In [ ]:
Bathy <- rast(bathy_file)
print("Converted into Raster File")

In [ ]:
# ==============================================================================
# 3. MAP PLOTTING ----
# ==============================================================================
png <- 1
if (png) {
  png(filename = "outputs/Fig1.png", width = 1080, height = 720)
    
} else {
  pdf("outputs/Fig1.pdf")
}

Depth_cuts <- c(-8200 ,-7000 ,-6000 ,-5000, -4000, -3000, -1800, -1400, -1000,  -600,  -400 , -200  ,   0 ,   50  , 250   ,500)
Depth_cols <- c(
  "#D6EAF8", "#AED6F1", "#85C1E9", "#5DADE2", "#3498DB",
  "#5DADE2", "#85C1E9", "#A9CCE3", "#D4E6F1", "#EBF5FB",
  "#F4F6F7", "#F8F9F9", "#FDFEFE", "#F2F3F4", "#EAEDED"
)

# Plot bathymetry
plot(Bathy,breaks = Depth_cuts, col = Depth_cols, legend = FALSE, axes = FALSE, box = FALSE,mar=c(0,0,0,0))

# Save the plot ---
dev.off()


In [ ]:
# Get time variable

# Temporal dimension 
time

# Number of time steps
nt <- dim(time)
nt


t_units <- ncatt_get(nc_file, "time", "units")
t_units

In [ ]:
# Get an ocean variable
T_array <- ncvar_get(nc_file,nc_file$var[[1]])
T_array

T <- nc_file$var[1]

#variable's attributes
ncatt_get(nc_file, T, "long_name")   #long name
ncatt_get(nc_file, T, "units")       #measure unit
fillvalue <- ncatt_get(nc_file, T, "_FillValue")  #(optional)  

In [ ]:
# convert time -- split the time units string into fields
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dminutes(time)
date

In [ ]:
# Quick map plot

# set the time step
t <- 1 #temperature on 2020-01-16
T_slice <- T_array[,,t]

image(longitude,latitude,T_slice, col = rev(brewer.pal(10,"RdBu")))

In [ ]:
grid <- expand.grid(lon=longitude, lat=latitude)  #create a set of lonxlat pairs of values, one for each element in the Temp_array

cutpts <- c(12,13,14,15,16,17,18,19,20)   # set colorbar
levelplot(T_slice ~ lon * lat,
          data=grid, region=TRUE,
          pretty=T, at=cutpts, cuts=9,
          col.regions=(rev(brewer.pal(9,"RdBu"))), contour=0,
          xlab = "Longitude", ylab = "Latitude",
          main = "Sea Water Potential Temperature (°C)"
          )

### Convert NetCDF to CSV
Based on [Copernicus Marine Services](https://help.marine.copernicus.eu/en/articles/6328012-how-to-convert-netcdf-to-csv-using-r)

In [ ]:
#  Extract the coordinates

dim_lon <- ncvar_get(nc_file, "lon")
dim_lat <- ncvar_get(nc_file, "lat")
dim_depth <- ncvar_get(nc_file, "z")

In [ ]:
t_units <- ncatt_get(nc_file, "time", "units")
t_ustr <- strsplit(t_units$value, " ")
t_dstr <- strsplit(unlist(t_ustr)[3], "-")
date <- ymd(t_dstr) + dseconds(dim_time)
date

In [ ]:
coords <- as.matrix(expand.grid(dim_lon, dim_lat, dim_depth, date))

In [ ]:
var1 <- ncvar_get(nc_ds, "var1", collapse_degen=FALSE)
var2 <- ncvar_get(nc_ds, "var2", collapse_degen=FALSE)

In [ ]:
nc_df <- data.frame(cbind(coords, var1, var2))
names(nc_df) <- c("lon", "lat", "depth", "time", "var1", "var2")
head(na.omit(nc_df), 5)  # Display some non-NaN values for a visual check
csv_fname <- "netcdf_filename.csv"
write.table(nc_df, csv_fname, row.names=FALSE, sep=";")